In [0]:
df_crm_cust = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_customers
""")
df_crm_products = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_products
""")
df_crm_sales = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.crm_sales
""")
df_erp_categories = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_categories
""")
df_erp_customers = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_customers
""")
df_erp_locations = spark.sql("""
SELECT * FROM sqldatawarehouse.bronze.erp_erp_locations
""")

### Add metadata columns

In [0]:
# from pyspark.sql.functions import current_timestamp

# df_crm_cust = df_cust.withColumn('dwh_created_date', current_timestamp())
# df_crm_products = df_crm_products.withColumn('dwh_created_date', current_timestamp())
# df_crm_sales = df_crm_sales.withColumn('dwh_created_date', current_timestamp())
# df_erp_categories = df_erp_categories.withColumn('dwh_created_date', current_timestamp())
# df_erp_customers = df_erp_customers.withColumn('dwh_created_date', current_timestamp())
# df_erp_locations = df_erp_locations.withColumn('dwh_created_date', current_timestamp())

df_crm_cust.display()



### `Check` for Nulls or Duplicates in Primary Key

#### Expectation: No Result

In [0]:
from pyspark.sql import functions as F

result = (
    df_crm_cust.groupBy("cst_id")
      .count()
      .filter((F.col("count") > 1) | F.col("cst_id").isNull())
)

result.show()

### Clear Null and Duplicates

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cust = (
    df_crm_cust.filter(
        F.col("cst_id").isNotNull() &
        (F.trim(F.col("cst_id")) != "")
    )
)

w = Window.partitionBy("cst_id").orderBy(F.desc("cst_create_date"))

df_crm_cust = (
    df_crm_cust
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

### Check for unwanted spaces in string values

In [0]:
df = df_crm_cust.where(F.col('cst_lastname') != F.trim(F.col("cst_lastname")))
display(df)

In [0]:
df_crm_cust = df_crm_cust.withColumn('cst_firstname', F.trim(F.col('cst_firstname')))
df_crm_cust = df_crm_cust.withColumn('cst_lastname', F.trim(F.col('cst_lastname')))


### Data Standardization & consistency

In [0]:
display(df_crm_cust.select('cst_gndr').distinct())

In [0]:
df_crm_cust = df_crm_cust.withColumn(
    "cst_gndr",
    F.when(F.trim(F.upper(F.col("cst_gndr"))).isNull(), "Unknown")
     .when(F.trim(F.upper(F.col("cst_gndr"))) == "M", "Male")
     .when(F.trim(F.upper(F.col("cst_gndr"))) == "F", "Female")
     .otherwise("Unknown")
)
display(df_crm_cust.select("cst_gndr").distinct())

In [0]:
display(df_crm_cust.select('cst_marital_status').distinct())

In [0]:
df_crm_cust = df_crm_cust.withColumn(
    "cst_marital_status",
    F.when(F.trim(F.upper(F.col("cst_marital_status"))).isNull(), "Unknown")
     .when(F.trim(F.upper(F.col("cst_marital_status"))) == "M", "Married")
     .when(F.trim(F.upper(F.col("cst_marital_status"))) == "S", "Single")
     .otherwise("Unknown")
)
display(df_crm_cust.select("cst_marital_status").distinct())

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

# Silver layer configuration
SILVER_DB = "sqldatawarehouse.silver"
AUDIT_TABLE = f"{SILVER_DB}.transformation_audit"

# Generate batch ID for this silver layer transformation
batch_id = str(uuid.uuid4())

In [0]:
def log_silver_audit(source_table, target_table, row_count, transformations_applied, status):
    """
    Log silver layer transformation to audit table
    
    Args:
        source_table: Bronze table(s) used as source
        target_table: Silver table written to
        row_count: Number of rows in the result
        transformations_applied: Description of transformations
        status: SUCCESS or FAILED
    """
    audit_df = spark.createDataFrame([(
        source_table,
        target_table,
        row_count,
        batch_id,
        transformations_applied,
        status,
        datetime.now()
    )], [
        "source_table",
        "target_table",
        "row_count",
        "batch_id",
        "transformations_applied",
        "status",
        "transformation_time"
    ])
    
    audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)

In [0]:
# Define source and target
source_table = "sqldatawarehouse.bronze.crm_customers"
target_table = f"{SILVER_DB}.crm_customers"

# List of transformations applied
transformations = [
    "Removed null/empty cst_id",
    "Deduplicated by cst_id (keeping latest by cst_create_date)",
    "Trimmed whitespace from cst_firstname, cst_lastname",
    "Standardized cst_gndr to Male/Female/Unknown",
    "Standardized cst_marital_status to Married/Single/Unknown"
]

try:
    # Get row count
    row_count = df_crm_cust.count()
    
    # Write to silver layer
    df_crm_cust.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)
    
    print(f"✓ Successfully wrote {row_count} rows to {target_table}")
    
    # Log audit
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=row_count,
        transformations_applied="; ".join(transformations),
        status="SUCCESS"
    )
    
    print(f"✓ Audit logged to {AUDIT_TABLE}")
    
except Exception as e:
    print(f"✗ Failed to write to {target_table}: {str(e)}")
    
    # Log failure
    log_silver_audit(
        source_table=source_table,
        target_table=target_table,
        row_count=0,
        transformations_applied="; ".join(transformations),
        status=f"FAILED: {str(e)[:200]}"
    )
    
    raise

In [0]:
# Verify the table was created
print(f"\nVerifying {target_table}:")
spark.sql(f"DESCRIBE TABLE {target_table}").show(truncate=False)

print(f"\nRow count in {target_table}: {spark.table(target_table).count()}")

# Show recent audit logs
print(f"\nRecent audit logs from {AUDIT_TABLE}:")
spark.sql(f"""
    SELECT 
        target_table,
        row_count,
        status,
        transformation_time,
        transformations_applied
    FROM {AUDIT_TABLE}
    ORDER BY transformation_time DESC
    LIMIT 5
""").display(truncate=False)